In [0]:
order_silver = spark.table("02_silver_catalog.silver.order")
invoice_silver = spark.table("02_silver_catalog.silver.invoice")
estimate_silver = spark.table("02_silver_catalog.silver.estimate")
customer_survey_silver = spark.table("02_silver_catalog.silver.customer_survey")
store_silver = spark.table("02_silver_catalog.silver.store")
ns_budget_silver = spark.table("02_silver_catalog.silver.ns_budget")

In [0]:
from pyspark.sql import functions as F

order_df = order_silver.alias("o")

fact_order_cycle = (
    order_df
    .withColumn("vehicle_in_ts", F.col("vehicle_in_datetime").cast("timestamp"))
    .withColumn("vehicle_out_ts", F.col("vehicle_out_datetime").cast("timestamp"))
    .withColumn("work_start_ts", F.col("actual_work_start_datetime").cast("timestamp"))
    .withColumn("work_complete_ts", F.col("actual_completion_datetime").cast("timestamp"))
    .withColumn("delivery_ts", F.col("actual_delivery_datetime").cast("timestamp"))
    
    # Stage durations in days
    .withColumn("days_vehicle_in_to_work_start",
                F.datediff("work_start_ts", "vehicle_in_ts"))
    .withColumn("days_work_start_to_completion",
                F.datediff("work_complete_ts", "work_start_ts"))
    .withColumn("days_completion_to_delivery",
                F.datediff("delivery_ts", "work_complete_ts"))
    .withColumn("total_days_in_shop",
                F.datediff("delivery_ts", "vehicle_in_ts"))

    # Completion accuracy
    .withColumn("planned_vs_actual_completion_days",
                F.datediff("actual_completion_datetime", "planned_completion_datetime"))

    # Delivery accuracy
    .withColumn("promised_vs_actual_delivery_days",
                F.datediff("actual_delivery_datetime", "promised_delivery_datetime"))

    # Surrogate key (optional)
    .withColumn("order_key", F.monotonically_increasing_id())
)

# Write to Delta Table (Gold Layer)
# fact_order_cycle.write.format("delta").mode("overwrite").saveAsTable("gold.fact_order_cycle")

In [0]:
invoice_df = invoice_silver.alias("i")
order_df = order_silver.alias("o")

fact_invoice = (
    invoice_df
    .join(order_df, "order_id", "left")
    .select(
        F.monotonically_increasing_id().alias("invoice_key"),
        "i.invoice_id",
        "i.order_id",
        "o.store_id",
        "o.technician_id",
        "invoice_amount",
        "payment_mode",
        "currency",
        "invoice_date",
        "invoice_ts"
    )
)

# fact_invoice.write.format("delta").mode("overwrite").saveAsTable("gold.fact_invoice")

In [0]:
survey_df = customer_survey_silver.alias("s")
order_df = order_silver.alias("o")

fact_survey = (
    survey_df
    .join(order_df, "order_id", "left")
    .select(
        F.monotonically_increasing_id().alias("survey_key"),
        "s.survey_id",
        "s.order_id",
        "o.store_id",
        "o.technician_id",
        "responded_flag",
        "survey_sent_date",
        "survey_response_date",
        "overall_satisfaction_rating",
        "cleanliness_rating",
        "communication_rating",
        "work_quality_rating",
        "delivered_on_time_rating",
        F.datediff("survey_response_date", "survey_sent_date").alias("days_to_respond")
    )
)

# fact_survey.write.format("delta").mode("overwrite").saveAsTable("gold.fact_survey")

### Dimensional tables

In [0]:
from pyspark.sql import functions as F

dim_store = (
    store_silver
    .select(
        F.monotonically_increasing_id().alias("store_key"),
        "store_id",
        "store_name",
        "city",
        "state",
        "manager_id",
        "manager_name",
        "opened_year",
        "store_type"
    )
)

# dim_store.write.format("delta").mode("overwrite").saveAsTable("gold.dim_store")

In [0]:
dim_technician = (
    order_silver
    .select("technician_id", "technician_name")
    .distinct()
    .withColumn("technician_key", F.monotonically_increasing_id())
    .select(
        "technician_key",
        "technician_id",
        "technician_name"
    )
)

# dim_technician.write.format("delta").mode("overwrite").saveAsTable("gold.dim_technician")

In [0]:
dim_estimator = (
    estimate_silver
    .select("estimator_id", "estimator_name")
    .distinct()
    .withColumn("estimator_key", F.monotonically_increasing_id())
    .select(
        "estimator_key",
        "estimator_id",
        "estimator_name"
    )
)

# dim_estimator.write.format("delta").mode("overwrite").saveAsTable("gold.dim_estimator")

In [0]:
dim_vehicle = (
    order_silver
    .select("vehicle_no", "vehicle_make", "vehicle_model")
    .distinct()
    .withColumn("vehicle_key", F.monotonically_increasing_id())
    .select(
        "vehicle_key",
        "vehicle_no",
        "vehicle_make",
        "vehicle_model"
    )
)

# dim_vehicle.write.format("delta").mode("overwrite").saveAsTable("gold.dim_vehicle")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Get min & max date from order and invoice tables
min_date = order_silver.select(F.min("vehicle_in_date")).first()[0]
max_date = invoice_silver.select(F.max("invoice_date")).first()[0]

# Create sequence of dates
date_df = (
    spark.range(0, (max_date - min_date).days + 1)
    .withColumn("date", F.expr(f"date_add('{min_date}', id)"))
)

dim_date = (
    date_df
    .withColumn("date_key", F.date_format("date", "yyyyMMdd"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_of_week", F.date_format("date", "E"))
    .withColumn("is_weekend", F.col("day_of_week").isin("Sat", "Sun"))
)

# dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")

In [0]:
from pyspark.sql import functions as F

fact_order = fact_order_cycle.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_vehicle_df = dim_vehicle.alias("dv")
dim_date_df = dim_date.alias("dd")

# Join to store
fact_order_dim = (
    fact_order
    .join(dim_store_df, fact_order.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_order.technician_id == dim_technician_df.technician_id, "left")
    .join(dim_vehicle_df, fact_order.vehicle_no == dim_vehicle_df.vehicle_no, "left")

    # Join date dimensions
    .join(dim_date_df.withColumnRenamed("date_key", "vehicle_in_date_key"),
          fact_order.vehicle_in_date == dim_date_df.date, "left")

    .join(dim_date_df.withColumnRenamed("date_key", "delivery_date_key"),
          fact_order.actual_delivery_datetime.cast("date") == dim_date_df.date, "left")
)

# Select final fields with surrogate keys
fact_order_conformed = fact_order_dim.select(
    "order_key",
    "order_id",
    dim_store_df.store_key,
    dim_technician_df.technician_key,
    dim_vehicle_df.vehicle_key,
    "vehicle_in_date_key",
    "delivery_date_key",
    "service_type",
    "vehicle_in_datetime",
    "vehicle_out_datetime",
    "actual_work_start_datetime",
    "actual_completion_datetime",
    "actual_delivery_datetime",
    "days_vehicle_in_to_work_start",
    "days_work_start_to_completion",
    "days_completion_to_delivery",
    "total_days_in_shop",
    "planned_vs_actual_completion_days",
    "promised_vs_actual_delivery_days"
)

# Write back to Gold Layer
# fact_order_conformed.write.format("delta").mode("overwrite").saveAsTable("gold.fact_order_cycle_final")

In [0]:
from pyspark.sql import functions as F

fact_inv = fact_invoice.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_date_df = dim_date.alias("dd")

fact_invoice_dim = (
    fact_inv
    .join(dim_store_df, fact_inv.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_inv.technician_id == dim_technician_df.technician_id, "left")
    .join(dim_date_df.withColumnRenamed("date_key", "invoice_date_key"),
          fact_inv.invoice_date == dim_date_df.date, "left")
)

fact_invoice_conformed = fact_invoice_dim.select(
    "invoice_key",
    "invoice_id",
    "order_id",
    dim_store_df.store_key,
    dim_technician_df.technician_key,
    "invoice_amount",
    "payment_mode",
    "currency",
    "invoice_date_key",
    "invoice_ts"
)

# fact_invoice_conformed.write.format("delta").mode("overwrite").saveAsTable("gold.fact_invoice_final")

In [0]:
from pyspark.sql import functions as F

fact_s = fact_survey.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_date_df = dim_date.alias("dd")

fact_survey_dim = (
    fact_s
    .join(dim_store_df, fact_s.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_s.technician_id == dim_technician_df.technician_id, "left")

    .join(dim_date_df.withColumnRenamed("date_key", "survey_sent_date_key"),
          fact_s.survey_sent_date == dim_date_df.date, "left")

    .join(dim_date_df.withColumnRenamed("date_key", "survey_response_date_key"),
          fact_s.survey_response_date == dim_date_df.date, "left")
)

fact_survey_conformed = fact_survey_dim.select(
    "survey_key",
    "survey_id",
    "order_id",
    dim_store_df.store_key,
    dim_technician_df.technician_key,
    "responded_flag",
    "survey_sent_date_key",
    "survey_response_date_key",
    "overall_satisfaction_rating",
    "cleanliness_rating",
    "communication_rating",
    "work_quality_rating",
    "delivered_on_time_rating",
    "days_to_respond"
)

# fact_survey_conformed.write.format("delta").mode("overwrite").saveAsTable("gold.fact_survey_final")